# BingePlay Project
- Built by - Dibya

import sqlalchemy and pymysql

In [16]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

import os
from dotenv import load_dotenv

load_dotenv()  # reads credentials from a local .env file (not committed to GitHub)

url = URL.create(
    drivername="mysql+pymysql",
    username=os.getenv("DB_USER", "root"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST", "127.0.0.1"),
    port=int(os.getenv("DB_PORT", 3306)),
    database=os.getenv("DB_NAME", "bingeplay"),
)
engine = create_engine(url)

## Checking connection 


In [17]:
from sqlalchemy import create_engine, text

try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT 1"))
        print("✅ Connected! Test query result:", result.scalar())
except Exception as e:
    print("❌ Connection failed:", e)

✅ Connected! Test query result: 1


In [18]:
import pandas as pd

df = pd.read_sql(text("SHOW TABLES;"), engine)
print(df)

  Tables_in_bingeplay
0             ratings
1               shows
2       subscriptions
3               users
4      watch_sessions


# Q1 Active revenue

 Answer : 2340 total active subscription & total income is 784260


In [5]:
query1 = """
select plan, count(*), sum( monthly_price_inr) as total_income from subscriptions 
where status ='active' OR (end_date = '2026-06-30' OR end_date is null)
group by plan;
"""

df1 = pd.read_sql(query1, engine)
print(df1)

      plan  count(*)  total_income
0    Basic      1314      261486.0
1  Premium       648      258552.0
2   Family       378      264222.0


# Q2 - Signup momentum

Answer : June & May month has max signup with 600

In [6]:
query2 = """
SELECT 
MONTHNAME(signup_date) AS month,
COUNT(*) AS signup_count
FROM users
WHERE signup_date IS NOT NULL
GROUP BY MONTHNAME(signup_date), MONTH(signup_date)
ORDER BY MONTH(signup_date) desc;
"""

df2 = pd.read_sql(query2, engine)
print(df2)

      month  signup_count
0      June           600
1       May           600
2     April           550
3     March           500
4  February           400
5   January           350


# Q3 - Device analytics

It gives no of sessions, avg time , conpletion rate of each session

In [25]:
query3 ="""
SELECT 
device_type AS platform,
COUNT(*) AS total,
SUM(watch_minutes) AS time,
AVG(watch_minutes) AS avg_time,
SUM(completed) AS completed_sessions,
ROUND(100.0 * SUM(completed) / COUNT(*), 2) AS completion_rate_pct
FROM watch_sessions
where user_id is not null
GROUP BY device_type
ORDER BY total DESC;
"""

df3 = pd.read_sql(query3,engine)
print(df3)

  platform  total       time  avg_time  completed_sessions  \
0   Mobile  50172  1504355.0   29.9840             30223.0   
1       TV  27981   840595.0   30.0416             16783.0   
2   Laptop  15105   453434.0   30.0188              9140.0   
3   Tablet   7091   210733.0   29.7184              4240.0   

   completion_rate_pct  
0                60.24  
1                59.98  
2                60.51  
3                59.79  


# Q4 - Rating distribution
It shows review rating and % rating

 -> What percentage of all ratings are 4 or 5 stars?
- Answer : 71.34%

In [26]:
query4 = """
SELECT 
stars AS rated_star,
COUNT(*) AS ratings,
ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS percentage_of_rating
FROM ratings
GROUP BY stars
ORDER BY stars desc;
"""

df4 = pd.read_sql(query4,engine)
print(df4)

   rated_star  ratings  percentage_of_rating
0           5     1786                 35.72
1           4     1781                 35.62
2           3      847                 16.94
3           2      352                  7.04
4           1      234                  4.68


# Q5 - Originals vs acquired
comparison between 2 subscription model Originals and normal

-> which group performs better on ratings, and by how much?
 - Answer : BingePlay Originals perform better as compared to Acquried shows . it is 19.46% better rated than Acquried shows .

In [27]:
query5 = """
SELECT
CASE WHEN is_original = 1 THEN 'BingePlay Originals' 
ELSE 'Acquired' 
END AS content_type,
COUNT(*) AS num_shows,
ROUND(AVG(imdb_rating), 2) AS avg_imdb_rating,
ROUND(AVG(release_year), 2) AS avg_release_year
FROM shows
GROUP BY is_original
ORDER BY is_original DESC;
"""

df5 = pd.read_sql(query5,engine)
print(df5)

          content_type  num_shows  avg_imdb_rating  avg_release_year
0  BingePlay Originals         30             7.92           2020.37
1             Acquired         70             6.63           2020.73


# Q6 -Binge day detection

- Total number of binge days that occurred
- Answer : 414
- The user_id who had the MOST binge days in Q2 2024
- Answer :U02956
- How many binge days that user had
- Answer : v

In [28]:
query6A = """
WITH binge_days AS (
    SELECT 
    user_id,
    show_id,
    DATE(session_date) AS binge_date,
    COUNT(*) AS session_count
    FROM watch_sessions
    WHERE session_date >= '2024-04-01' AND session_date < '2024-07-01'
    GROUP BY user_id, show_id, DATE(session_date)
    HAVING COUNT(*) >= 5
)
SELECT COUNT(*) AS total_binge_days
FROM binge_days;
"""

df6A = pd.read_sql(query6A, engine)
print(df6A)

   total_binge_days
0               414


In [29]:
query6B = """
WITH binge_days AS (
    SELECT 
    user_id,
    show_id,
    DATE(session_date) AS binge_date,
    COUNT(*) AS session_count
    FROM watch_sessions
    WHERE session_date >= '2024-04-01' AND session_date < '2024-07-01'
    GROUP BY user_id, show_id, DATE(session_date)
    HAVING COUNT(*) >= 5
)
SELECT user_id, COUNT(*) AS user_binge_days
FROM binge_days
GROUP BY user_id
ORDER BY user_binge_days DESC
LIMIT 1;
"""

df6B = pd.read_sql(query6B, engine)
print(df6B)

  user_id  user_binge_days
0  U02956                8


# Q7 - Q1 signups who never watched 
- Total number of Q1 signups
- Answer :1250
- Number of Q1 signups who have never watched anything
- Answer : 226


In [30]:
query7 = """
SELECT 
    (SELECT COUNT(*) 
     FROM users 
     WHERE signup_date >= '2024-01-01' AND signup_date < '2024-04-01') AS Q1_signups,
     
    (SELECT COUNT(*) 
     FROM users u
     LEFT JOIN watch_sessions ws ON u.user_id = ws.user_id
     WHERE u.signup_date >= '2024-01-01' AND u.signup_date < '2024-04-01'
       AND ws.session_id IS NULL) AS never_watched;
"""

df7 = pd.read_sql(query7, engine)
print(df7)

   Q1_signups  never_watched
0        1250            226


# Q8- The over-paying Premium/Family users
- Number of Premium/Family users whose entire watch history is Basic-tier only
- Answer : 9

In [31]:
query8 ="""
WITH ranked_subs AS (
    SELECT 
    user_id, 
    plan,
    ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY start_date DESC) AS rn
    FROM subscriptions
    WHERE end_date IS NULL OR end_date >= '2024-06-30'
),
current_plan AS (
    SELECT user_id, plan
    FROM ranked_subs
    WHERE rn = 1
)
SELECT COUNT(*) AS overpaying_users
FROM current_plan cp
WHERE cp.plan IN ('Premium', 'Family') AND EXISTS (
-- user must have watched at least something
SELECT 1 FROM watch_sessions ws WHERE ws.user_id = cp.user_id
  )
  AND NOT EXISTS (
      -- no show they watched required more than Basic
      SELECT 1
      FROM watch_sessions ws
      JOIN shows sh ON ws.show_id = sh.show_id
      WHERE ws.user_id = cp.user_id
        AND sh.min_plan IN ('Premium', 'Family')
  );
"""

pd.read_sql(query8, engine)

,overpaying_users
0,9


# Q9 - Upgrade success cohort

- Number of such users
- Answer :55
- Their average days from signup to first upgrade
- Answer :64.96
- 

In [32]:
query9 = """
WITH ranked_subs AS (
    SELECT 
    user_id,
    plan,
    start_date,
    end_date,
    ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY start_date ASC) AS rn_asc
    FROM subscriptions
),
first_sub AS (
    -- each user's earliest subscription record
    SELECT user_id, plan AS first_plan, start_date AS first_start_date
    FROM ranked_subs
    WHERE rn_asc = 1
),
first_upgrade AS (
    -- earliest Premium/Family record that comes AFTER their first (Basic) subscription
    SELECT 
    rs.user_id,
    MIN(rs.start_date) AS first_upgrade_date
    FROM ranked_subs rs
    JOIN first_sub fs ON rs.user_id = fs.user_id
    WHERE rs.plan IN ('Premium', 'Family') AND rs.start_date > fs.first_start_date
    GROUP BY rs.user_id
),
current_active AS (
    -- most recent subscription row still covering 30-Jun-2024
    SELECT user_id, plan
    FROM (
        SELECT 
        user_id, plan, start_date, end_date,
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY start_date DESC) AS rn_desc
        FROM subscriptions
        WHERE start_date <= '2024-06-30' AND (end_date IS NULL OR end_date >= '2024-06-30')
    ) x
    WHERE rn_desc = 1
)
SELECT 
COUNT(*) AS upgrade_success_count,
ROUND(AVG(DATEDIFF(fu.first_upgrade_date, u.signup_date)), 2) AS avg_days_to_upgrade
FROM users u
JOIN first_sub fs ON u.user_id = fs.user_id
JOIN first_upgrade fu ON u.user_id = fu.user_id
JOIN current_active ca ON u.user_id = ca.user_id
WHERE u.signup_date >= '2024-01-01' AND u.signup_date < '2024-02-01' AND fs.first_plan = 'Basic';
"""

pd.read_sql(query9, engine)

,upgrade_success_count,avg_days_to_upgrade
0,55,64.96


# Q10 - Cliffhanger comebacks
- total event
- Answer : 4345

In [33]:
query_total = """
SELECT COUNT(*) AS total_cliffhanger_events
FROM (
    SELECT DISTINCT a.user_id, a.show_id, DATE(a.session_date) AS incomplete_date
    FROM watch_sessions a
    JOIN watch_sessions b ON a.user_id = b.user_id
        AND a.show_id = b.show_id AND a.completed = 0 AND DATE(b.session_date) BETWEEN DATE(a.session_date) + INTERVAL 1 DAY 
        AND DATE(a.session_date) + INTERVAL 7 DAY
) AS events;
"""

df_total = pd.read_sql(query_total, engine)
print(df_total)

   total_cliffhanger_events
0                      4345


In [34]:
query_top_show = """
SELECT sh.show_id, sh.title, COUNT(*) AS comeback_count
FROM (
    SELECT DISTINCT a.user_id, a.show_id, DATE(a.session_date) AS incomplete_date
    FROM watch_sessions a
    JOIN watch_sessions b ON a.user_id = b.user_id
        AND a.show_id = b.show_id AND a.completed = 0
        AND DATE(b.session_date) BETWEEN DATE(a.session_date) + INTERVAL 1 DAY 
                                      AND DATE(a.session_date) + INTERVAL 7 DAY
) AS events
JOIN shows sh ON events.show_id = sh.show_id
GROUP BY sh.show_id, sh.title
ORDER BY comeback_count DESC
LIMIT 1;
"""

df_top_show = pd.read_sql(query_top_show, engine)
print(df_top_show)

  show_id             title  comeback_count
0    S088  Rayalaseema Raga              64


# Q11 - Consecutive-week engagement


In [35]:
query11 = """
WITH user_weeks AS (
    SELECT DISTINCT
    user_id,
    FLOOR(DATEDIFF(DATE(session_date), '2000-01-03') / 7) AS week_ordinal
    FROM watch_sessions
),
ranked AS (
    SELECT 
        user_id,
        week_ordinal,
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY week_ordinal ASC) AS rn
    FROM user_weeks
),
streaks AS (
    SELECT 
    user_id,
    week_ordinal,
    rn,
    (week_ordinal - rn) AS streak_group
    FROM ranked
),
streak_lengths AS (
    SELECT 
    user_id,
    streak_group,
    COUNT(*) AS streak_weeks
    FROM streaks
    GROUP BY user_id, streak_group
),
top_streak AS (
    SELECT user_id, streak_weeks
    FROM streak_lengths
    ORDER BY streak_weeks DESC
    LIMIT 1
)
SELECT 
    (SELECT COUNT(DISTINCT user_id) FROM streak_lengths WHERE streak_weeks >= 4) AS users_with_4plus_streak,
    (SELECT streak_weeks FROM top_streak) AS longest_streak_weeks,
    (SELECT user_id FROM top_streak) AS user_with_longest_streak;
"""

df11 = pd.read_sql(query11, engine)
print(df11)

   users_with_4plus_streak  longest_streak_weeks user_with_longest_streak
0                     1675                    26                   U01793


# Q12 - Churn signal detection

In [24]:
query_churn = """
WITH monthly_totals AS (
    SELECT 
    user_id,
    SUM(CASE WHEN session_date >= '2024-05-01' AND session_date < '2024-06-01' 
                THEN watch_minutes ELSE 0 END) AS may_minutes,
    SUM(CASE WHEN session_date >= '2024-06-01' AND session_date < '2024-07-01' 
                THEN watch_minutes ELSE 0 END) AS june_minutes
    FROM watch_sessions
    WHERE session_date >= '2024-05-01' AND session_date < '2024-07-01'
    GROUP BY user_id
),
churn_calc AS (
    SELECT 
    user_id,
    may_minutes,
    june_minutes,
    ROUND((may_minutes - june_minutes) * 100.0 / may_minutes, 2) AS drop_percentage
    FROM monthly_totals
    WHERE may_minutes > 0
)
SELECT 
cc.user_id,
u.name,
cc.may_minutes AS may_2024_watch_minutes,
cc.june_minutes AS june_2024_watch_minutes,
cc.drop_percentage
FROM churn_calc cc
JOIN users u ON cc.user_id = u.user_id
WHERE cc.drop_percentage >= 50
ORDER BY cc.drop_percentage DESC;
"""

df_churn = pd.read_sql(query_churn, engine)
print(df_churn)
print(f"\nTotal churn signal users: {len(df_churn)}")

    user_id              name  may_2024_watch_minutes  \
0    U00023    Amit Mukherjee                    43.0   
1    U00166  Shaurya Malhotra                    94.0   
2    U00211        Ravi Menon                   336.0   
3    U00225     Kritika Patil                   209.0   
4    U00237    Shaurya Bansal                   126.0   
..      ...               ...                     ...   
516  U01858  Sanjay Mukherjee                   296.0   
517  U02530       Nandini Roy                   451.0   
518  U01192   Rohit Mukherjee                   136.0   
519  U01806      Yuvraj Raman                    78.0   
520  U02100      Vikram Singh                   204.0   

     june_2024_watch_minutes  drop_percentage  
0                        0.0           100.00  
1                        0.0           100.00  
2                        0.0           100.00  
3                        0.0           100.00  
4                        0.0           100.00  
..                       ..

# AI Assistance Disclosure
This project was implemented by me using Python, Pandas, and NumPy.

- I used ChatGPT only for:

  ->Syntax guidance

  ->Debugging

  ->Code review

- All outputs, spending percentages, anomaly detection results, archetype detection, and final conclusions were generated by running the code on the provided dataset.